# 2HRX9P6HKXA8V

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf


# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import plot_time_series, plot_time_series_plant_based
from tools.labeling_functions import plot_dish_time_series, dish_time_series, fully_relabel_and_consolidate, rename_items

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

%store -r time_differences
%store -r time_differences_details
# import from pickle
if 'time_differences' not in locals():
    time_differences = pd.read_pickle('data/2_palate_data_parquet_cleaned/time_differences.pkl')
    time_differences_details = pd.read_pickle('data/2_palate_data_parquet_cleaned/time_differences_details.pkl')
%store time_differences
%store time_differences_details

loc_id = '2HRX9P6HKXA8V'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Beyond")')['item_quantity'].sum())
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Beyond")')['item_quantity'].sum())

plot_time_series_plant_based('2HRX9P6HKXA8V', 
                 df_uncleaned.query('item_name.str.contains("Beyond")'), 
                 before_after_details_true, 
                 freq='W', 
                 subset=False)
plt.show()

In [ ]:
# display(time_differences_details[loc_id])
# display(time_differences[loc_id])

# # # 10 Hour Difference
# # df['item_name'].value_counts().sort_values(ascending=False).head(10)
# # df.loc[pd.Timestamp('2019-07-17 4:45:58+00:00'):pd.Timestamp('2019-07-18').tz_localize('UTC')].head(5)

# # # 20 Hour Difference
# # df.loc[pd.Timestamp('2020-06-07 22:27:20+00:00'):pd.Timestamp('2020-06-09').tz_localize('UTC')].head(10)
# # time_differences_details['2HRX9P6HKXA8V'][0][time_differences_details['2HRX9P6HKXA8V'][0] == 5]

In [ ]:
name_changes = {
    #"Veggie Wurst": ["Vegetarian"],
    "Potato Chips": ["Tim's Cascade Potato Chips", "Tim'S Cascade Potato Chips", "Kettle Brand Potato Chips", "Potato Chips", "Chips"], # 
    "Big Bob Bratwurst": ["Big Bob"],
    "Warm Bavarian Pretzel": ["Bavarian Pretzel", "Pretzel"],
    "Hans Jalapeno & Cheddar": ["Han's Jalapeno & Cheddar", "Jalapeno & Cheddar", "Jalapeño & Cheddar", "Hans' Jalape√±O & Cheddar", "Jalape√±O & Cheddar"],
    "Dirtyface Beer Wurst": ["Beer Wurst"],
    "Spinach Organic Chicken": [], # "Chicken", "Organic Chicken"
    "Italian Organic Chicken" : [],
    "Organic Chicken Sausage" : [],
    "Helgas Giant Kelbassi": ["Helga's Giant Kelbassi", "Kelbassi", "Giant Kelbassi", "Helga'S Giant Kelbassi"],
    "Large Sauerkraut - 8Oz Bowl": ["Side Sauerkraut - 8Oz Bowl", "Side Saurkraut"],
    "Gluhwein": ["Glühwein"],
    "Turkey Dog": ["Organic Turkey Dog"],
    "Gingerbread Cookie": ["Haus Made Gingerbread Cookie"],
    "Big City Beef Frank" : ["Big City"],
    "Vegan Soup": ["House Vegan Soup"], # "Veggie Soup" 
    #"Egift Card": ["Gift Card", "Promotional $5 Gift Certificates", "Donation $5 Gift Certificates"],
    "Bottled Water": ["Athena Bottled Water"],
    "Pepsi Bottled Sodas": ["Pepsi Fountain", "Diet Pepsi Fountain", "Pepsi", "Diet Pepsi", "Pepsi Fountain Sodas", "Pepsi Bottled Sodas 20Oz"],
    "Dr. Pepper Fountain": ["Dr. Pepper"],
    "7-Up Fountain": ["7-Up", "-Up Fountain", "-Up"],
    "Mountain Dew Fountain": ["Mountain Dew"],
    "Rootbeer Fountain": ["Rootbeer"],
    "German Potato Salad": ["G.P.S"],
    "Veggie Wurst": ["Vegetarian"],
    
    
    "Icicle Premium Pilsner": [],
    "Dirtyface Amber Lager": ["Dirtyface Beer Wurst", "Dirtyface Amber Mustard", "To-Go Dirtyface Amber Single 16Oz Can", "To-Go Dirtyface Amber 22Oz Bottle", "To-Go Dirtyface Amber 4 Pack 16Oz Cans", "Bottled Dirtyface"],
    "Bootjack IPA": ["Bootjack Ipa", "To-Go Bootjack Ipa Single Can 12Oz", "To-Go Bootjack Ipa 6 Pack 12Oz"],
    "Alpenhaze Hazy IPA": ["Alpenhaze"],
    "Colchuck Raspberry Wheat": ["To-Go Colchuck Raspberry Wheat Single Can 16Oz", "To-Go Colchuck Raspberry Wheat 4 Pack 16Oz", "Raspberry Dark Persuasion"],
    "Dark Persuasion": ["Dark Persuasion Chocolate Cake Ale", "To-Go Dark Persuasion German Chocolate Cake Ale Single Can 12Oz", "To-Go Dark Persuasion German Chocolate Cake Ale 6 Pack 12Oz"],
    "Hofbräu Original": ["Hofbr√§U Original"],
    "Hofbräu Hefe Weizen": ["Hofbrau Hefeweizen", "Hofbr√§U Hefeweizen", "Drubru Hefeweizen"],
    "Hofbräu Dunkel": ["Hofbr√§U Dunkel", "Hofbr√§U Dunkle", "Hofbrau Dunkel"],
    "Yonder Vantage Semi-Sweet Cider": ["Trailbreaker Cider 12Oz Can", "To-Go Trailbreaker Cider 12Oz Can", "Trailbreaker Cider 12Oz Can - Dine-In", "To-Go Trailbreaker Cider 12Oz Can *Takeout Only*", "Pitcher Draft Cider"],
    "Quartet Bordeaux-Style Blend": ["Cellars Trio", "Cellars Quartet", "Cellars Trio Bottle"],
    "Montage": ["Eagle Creek Montage (Merlot)", "Eagle Creek Montage", "Eagle Creek Montage Bottle"],
    "Chardonnay": ["Milbrandt Chardonnay Bottle"],
    "Pinot Grigio": ["Eagle Creek Pinot Grigio Bottle"],
    "Riesling": ["Ryan Patrick Riesling Bottle"],
    "Gewürztraminer": ["Icicle Ridge Gewurztraminer Bottle"],
    "Rosé of Sangiovese": ["Kestrel Ros√© Bottle", "Maryhill Ros√©", "Maryhill Ros√© Bottle"],
    "Ghostfish Brewing Company": ["Ghostfish Gf Can 12Oz"],
    "Athletic Brewing IPA": ["Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can", "To-Go Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can"],
    "Athletic Golden Ale": ["Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can", "To-Go Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can"],
    "Bitburger Drive Pilsner": ["N/A Beer - Bitburger"],
    "Crosscut Pilsner": ["To-Go Crosscut Pilsner Single Can 16Oz", "To-Go Crosscut Pilsner 4 Pack 16Oz"],
    "Kickstand Citra Pale Ale": ["Kickstand Pale Ale", "To-Go Kickstand Pale Ale Single Can 12Oz", "To-Go Kickstand Pale Ale 6 Pack 12Oz"],
    "Timbertown Brown": [],
    "Snow Creek K√∂Lsch": [],
    "Leavenworth Festbier": [],
    "Pamm'S American Lager": [],
    "Knock Off Australian Lager": [],
    "Enchantments Hazy Ipa": ["To-Go Enchantments Hazy Ipa Single Can 16Oz", "To-Go Enchantments Hazy Ipa Single Can 12Oz", "To-Go Enchantments Hazy Ipa 4 Pack 16Oz", "To-Go Enchantments Hazy Ipa 6 Pack 12Oz"],
    "One In Eight Fresh Hop Ipa": [],
    "Drumfire Dark Lager": [],
    "Drubru Kolsch": ["Drubru K√∂Lsch"],
    "Icicle Lager": [],
    "Gluten Free Beer": ["Gluten Free 16Oz - Dine-In", "To-Go Gluten Free 16Oz"],
    "N/A Beer": [],
    "Drubru Hefeweizen": ["Dru Bru K√∂Lsch"],
    "Sawdog IPA": ["Sawdog"],
    "Ryan Patrick Riesling Bottle": ["Riesling"],
}

alcohol_changes = {
    "Icicle Premium Pilsner": [],
    "Dirtyface Amber Lager": ["Dirtyface Beer Wurst", "Dirtyface Amber Mustard", "To-Go Dirtyface Amber Single 16Oz Can", "To-Go Dirtyface Amber 22Oz Bottle", "To-Go Dirtyface Amber 4 Pack 16Oz Cans", "Bottled Dirtyface"],
    "Bootjack IPA": ["Bootjack Ipa", "To-Go Bootjack Ipa Single Can 12Oz", "To-Go Bootjack Ipa 6 Pack 12Oz"],
    "Alpenhaze Hazy IPA": ["Alpenhaze"],
    "Colchuck Raspberry Wheat": ["To-Go Colchuck Raspberry Wheat Single Can 16Oz", "To-Go Colchuck Raspberry Wheat 4 Pack 16Oz", "Raspberry Dark Persuasion"],
    "Dark Persuasion": ["Dark Persuasion Chocolate Cake Ale", "To-Go Dark Persuasion German Chocolate Cake Ale Single Can 12Oz", "To-Go Dark Persuasion German Chocolate Cake Ale 6 Pack 12Oz"],
    "Hofbräu Original": ["Hofbr√§U Original"],
    "Hofbräu Hefe Weizen": ["Hofbrau Hefeweizen", "Hofbr√§U Hefeweizen", "Drubru Hefeweizen"],
    "Hofbräu Dunkel": ["Hofbr√§U Dunkel", "Hofbr√§U Dunkle", "Hofbrau Dunkel"],
    "Yonder Vantage Semi-Sweet Cider": ["Trailbreaker Cider 12Oz Can", "To-Go Trailbreaker Cider 12Oz Can", "Trailbreaker Cider 12Oz Can - Dine-In", "To-Go Trailbreaker Cider 12Oz Can *Takeout Only*", "Pitcher Draft Cider"],
    "Quartet Bordeaux-Style Blend": ["Cellars Trio", "Cellars Quartet", "Cellars Trio Bottle"],
    "Montage": ["Eagle Creek Montage (Merlot)", "Eagle Creek Montage", "Eagle Creek Montage Bottle"],
    "Chardonnay": ["Milbrandt Chardonnay Bottle"],
    "Pinot Grigio": ["Eagle Creek Pinot Grigio Bottle"],
    "Riesling": ["Ryan Patrick Riesling Bottle"],
    "Gewürztraminer": ["Icicle Ridge Gewurztraminer Bottle"],
    "Rosé of Sangiovese": ["Kestrel Ros√© Bottle", "Maryhill Ros√©", "Maryhill Ros√© Bottle"],
    "Ghostfish Brewing Company": ["Ghostfish Gf Can 12Oz"],
    "Athletic Brewing IPA": ["Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can", "To-Go Athletic Brewing Ipa *Non-Alcoholic* 12Oz Can"],
    "Athletic Golden Ale": ["Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can", "To-Go Athletic Brewing Blonde Ale *Non-Alcoholic & Gluten Free* 12Oz Can"],
    "Bitburger Drive Pilsner": ["N/A Beer - Bitburger"],
    "Crosscut Pilsner": ["To-Go Crosscut Pilsner Single Can 16Oz", "To-Go Crosscut Pilsner 4 Pack 16Oz"],
    "Kickstand Citra Pale Ale": ["Kickstand Pale Ale", "To-Go Kickstand Pale Ale Single Can 12Oz", "To-Go Kickstand Pale Ale 6 Pack 12Oz"],
    "Timbertown Brown": [],
    "Snow Creek K√∂Lsch": [],
    "Leavenworth Festbier": [],
    "Pamm'S American Lager": [],
    "Knock Off Australian Lager": [],
    "Enchantments Hazy Ipa": ["To-Go Enchantments Hazy Ipa Single Can 16Oz", "To-Go Enchantments Hazy Ipa Single Can 12Oz", "To-Go Enchantments Hazy Ipa 4 Pack 16Oz", "To-Go Enchantments Hazy Ipa 6 Pack 12Oz"],
    "One In Eight Fresh Hop Ipa": [],
    "Drumfire Dark Lager": [],
    "Drubru Kolsch": ["Drubru K√∂Lsch"],
    "Icicle Lager": [],
    "Gluten Free Beer": ["Gluten Free 16Oz - Dine-In", "To-Go Gluten Free 16Oz"],
    "N/A Beer": [],
    "Drubru Hefeweizen": ["Dru Bru K√∂Lsch"],
    "Sawdog IPA": ["Sawdog"],
    "Ryan Patrick Riesling Bottle": ["Riesling"],
    "Tres Hombres":[],
    "37 Cellars Quartet":[],
    "37 Cellars Trio":[],
    "Ibc 4 Pack Cans 16Oz":[], "Ibc 6 Pack Cans":[], "Ibc 6 Pack Cans 12Oz":[], "To-Go Single Cans Ibc 16Oz":[], "To-Go Single Cans Ibc 12Oz":[]
}

# Item names to swap based on modications
modification_name_changes = [('Beyond Sausage', 'Carne', 'Meat Beyond Sausage'), # With Chile Con Carne
                             ('Beyond Sausage', 'Cream|Cheese|Mayo|Beech', 'Vegetarian Beyond Sausage'), #'Beyond Sausage With Dairy'
                             ('Veggie Wurst', 'Vegan', 'Vegan Veggie Wurst'), # Vegan Veggie Wurst
                             ('Veggie Wurst', 'Carne', 'Meat Veggie Wurst'), # Veggie Wurst With Chile Con Carne
                             ('Vegetarian', 'Vegan', 'Vegan Vegetarian'), #Vegan Vegetarian
                             ('Vegetarian', 'Carne', 'Meat Vegetarian'), # With Chile Con Carne
                             ('Vegan Chili', 'Cream|Cheese|Beech', 'Vegetarian Vegan Chili'),
                             ('Veggie Soup', 'Cream|Cheese|Beech', 'Vegetarian Veggie Soup'),
                             ('Potato Chips', 'Cheddar', 'Vegetarian Potato Chips')]

modification_name_changes_2 = []

non_alcoholic_drinks = [
    "Lemonade",
    "Pepsi Bottled Sodas",
    "Bottled Water",
    "Iced Tea",
    "Hot Cocoa",
    "Dr. Pepper Fountain",
    "Hot Tea",
    "Rootbeer Fountain",
    "7-Up Fountain",
    "Apple Juice",
    "Coffee",
    "Mountain Dew Fountain",
    "Pepsi Fountain Sodas 22Oz",
    "Gatorade Fountain",
    "Bottled Soda",
    "Gatorade",
    "Common Ground Coffee Amber",
    "Tap Water To-Go",
    "Fountain Refill"
]

merch = ["Souvenir Water Bottle",
         "Souvenir Pint Glass",
         "Souvenir Wine Glass",
         "Gift Card",
         "Royal Blue Tee",
         "Black Tee",
         "Trucker Hat",
         "Dog Cookie (Dog Treat)",
         "Egift Card",
         "Black Beanie Winter Hat",
         "Blue Zip Up Sweatshirt",
         "T-Shirt Blue *Sale* Limited Sizes",
         "Donation $5 Gift Certificates",
         "Keychain Bottle Opener",
         "Reusable Straw",
         "Re-Useable Straw",
         "Promotional $5 Gift Certificates",
         "Ben Davis Button Up",
         "Shipping Charge",
         "Cowbell",
         "Corkage Fee",
         'Magnet', 
         'Carryout Paper Bag', 
         'Reusable Tote Bag', 
         'Sticker',
         'Winter Pom Pom Hat', 
         'Gray Pullover Sweatshirt',
         'V-Neck Tee', 
         'Face Buff', 
         'Winter Hat',
         'Sleeve Baseball Tee - Black On Black',
         'Black/Gray Zip Up Sweatshirt',
         'Gray Pull Sweatshirt',
         'Blue Waffle Beanie Winter Hat',
         'Keychain',
         'T-Shirt Blue',
         'Scarf',
         'Blue Zip Sweatshirt',
         'Sleeve Baseball Tee - Black On Gray', 
         'Dog Treat',
         'Foodles', 
         'Black Winter Hat', 
         'Mountain Equipment Jacket',
         'Patch Logo', 
         'To-Go Refill', 
         'Black Beanie',
         'Christmas Sweater 20',
         'Mountain Equipment Vest',
         '3/4 Sleeve Baseball Tee - Black On Black',
         '3/4 Sleeve Baseball Tee - Black On Gray',
]

rare = df_uncleaned['item_name'].value_counts().to_frame('count').query('count < 10').index.tolist()

unknown = []

meat = []

vegetarian = ['Warm Bavarian Pretzel', 
              'Beyond Sausage With Dairy',
              'Vegetarian', 
              'Veggie Wurst', 
              'Vegetarian Chili', 
              'Gingerbread Cookie',
              'Carrots & Ranch',
              'Cheddar Potato Chips',
              'Vegetarian Soup']

vegan = ['Vegan Vegetarian', 
         'Vegan Veggie Wurst', 
         'Beyond Sausage',
         'Veggie Soup', 
         'Vegan Soup',
         'Vegan Chili',
         'Apple Slices',
         'Potato Chips', 
         'Large Sauerkraut - 8Oz Bowl']

alcoholic_drinks = list(alcohol_changes.keys())

others = {}

df_relabeled = fully_relabel_and_consolidate(
    df_uncleaned, 
    name_changes = name_changes,
    modification_name_changes = modification_name_changes, 
    vegan_list = vegan, 
    vegetarian_list = vegetarian, 
    meat_list = meat, 
    alcohol_list = alcoholic_drinks, 
    drinks_list = non_alcoholic_drinks, 
    merch = merch, 
    rare = rare, 
    unknown = unknown, 
    remove_categories = ["Merch","Drink","Rare","Alcohol"]
    )

df_relabeled.to_parquet(f"data/4_palate_data_parquet_relabeled/relabeled/{loc_id}_sales_and_menu.parquet")

menu_changes = {
    "Beyond Sausage": ["Meat Beyond Sausage","Vegetarian Beyond Sausage"],
    "Veggie Wurst": ["Meat Veggie Wurst", "Vegan Veggie Wurst"],
    "Vegan Chili": ["Vegetarian Vegan Chili"],
    "Potato Chips": ["Vegetarian Potato Chips"],
    "Veggie Soup": ["Meat Veggie Soup", "Vegetarian Veggie Soup"],
    "Vegetarian": ["Meat Vegetarian", "Vegan Vegetarian"]
    }

df_consolidated = df_relabeled.pipe(rename_items, name_changes = menu_changes)

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_min=7.75, legend_max=18.70, shift_adjustment=1/15)

df_consolidated.to_parquet(f"data/4_palate_data_parquet_relabeled/consolidated/{loc_id}_sales_and_menu.parquet")

dish_time_series(df_consolidated).to_csv(f"labeling/timelines/{loc_id}.csv")

In [ ]:
deepresearch.columns

In [ ]:
import matplotlib.dates as mdates
def plot_boolean_time_series(df, loc_id, before_after_details_true, dish_order, unmatched_items):
    fig, ax = plt.subplots(figsize=(14, 8))
    promo_dt = before_after_details_true.loc[loc_id,'cross_over_date'].tz_convert('UTC')

    cols_to_use = [col for col in dish_order if col in df.columns]
    reordered_df = df[cols_to_use]

    stacked = reordered_df.stack()
    df_true = stacked[stacked]

    dish_to_y = {dish: i for i, dish in enumerate(cols_to_use)}
    y_labels = df_true.index.get_level_values(1)
    y_coords = y_labels.map(dish_to_y).astype(float)

    x_min_times = df_true.index.get_level_values(0).start_time
    x_max_times = df_true.index.get_level_values(0).end_time

    # Create mask for unmatched items
    is_unmatched = y_labels.isin(unmatched_items)

    # Plot matched items
    ax.hlines(y=y_coords[~is_unmatched],
              xmin=x_min_times[~is_unmatched],
              xmax=x_max_times[~is_unmatched],
              lw=10,
              color='tab:blue')

    # Plot unmatched items
    ax.hlines(y=y_coords[is_unmatched],
              xmin=x_min_times[is_unmatched],
              xmax=x_max_times[is_unmatched],
              lw=10,
              color='grey') 

    ax.axvline(promo_dt, color='red', linestyle='--', alpha=0.5)

    ax.set_yticks(range(len(cols_to_use)))
    ax.set_yticklabels(cols_to_use)
    ax.invert_yaxis()

    for label in ax.get_yticklabels():
        if label.get_text() in unmatched_items:
            label.set_color('grey')

    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    fig.autofmt_xdate()

    ax.set_title(f'Weekly Existence of Dishes for {loc_id}')
    ax.set_xlabel('Date'); ax.set_ylabel('Dish')
    fig.tight_layout(rect=[0, 0.03, 0.85, 0.97])
    plt.show()


# Define ordering and unmatched items outside the function
ordering = [
    'Big Bob Bratwurst', 'Warm Bavarian Pretzel', 'Hans’ Jalapeño & Cheddar',
    'Helga’s Giant Kelbassi', 'Big City Beef Frank', 'German Potato Salad',
    'Tim’s Cascade Chips', 'Veggie Wurst', 'Dirtyface Beer Wurst',
    'Organic Turkey Dog', 'Oma’s Weisswurst', 'Beyond Sausage', 'Potato Soup',
    'Chili con Carne', 'Bockwurst', 'Vegan Lentil Soup', 'Vegan Chili',
    'Organic Chicken & Apple Sausage', 'Spinach Organic Chicken Sausage',
    'Italian Organic Chicken Sausage', 'Mediterranean Chicken Sausage',
    'Curt’s Currywurst', 'Beyond Sausage Intro', 'COVID Closure',
    'COVID Limited', 'Currywurst Removed', 'Holiday Special',
    'New Chicken Sausage Intro', 'Oktoberfest', 'Weisswurst Intro'
]

unmatched = [
    'Beyond Sausage Intro', 'COVID Closure', 'COVID Limited',
    'Currywurst Removed', 'Holiday Special', 'New Chicken Sausage Intro',
    'Oktoberfest', 'Weisswurst Intro'
]

    
deepresearch = (pd.read_csv('data/Muchen Haus Timeline.csv', index_col=0)
                .pipe(lambda x: x.set_index(pd.to_datetime(x.index).tz_localize('UTC')).to_period('W')))

plot_boolean_time_series(deepresearch, loc_id, before_after_details_true, ordering, unmatched)
plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_min=7.75, legend_max=18.70, shift_adjustment=1/15)

In [ ]:
df_consolidated['item_name'].value_counts()

In [ ]:
print(df_consolidated
      .query('~vegetarian')
      ['unit_price']
      .mean())
print((df_consolidated
       .query('~vegetarian')
       ['item_name']
       .nunique()) / 
      (df_consolidated
       ['item_name']
       .nunique()))
plt.plot(df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         (df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique()) / 
         (df_consolidated
         .query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique()), 
         'o', 
         alpha=0.5)
plt.show()

In [ ]:
# # Covariate creation
# lookback_period = 1
# lookback_unit = 'D'

# def season_from_month(month):
#     return 'winter' if month in [12, 1, 2] else \
#            'spring' if month in [3, 4, 5] else \
#            'summer' if month in [6, 7, 8] else 'fall'

# def weighted_avg(window):
#     return (window['item_quantity'] * window['unit_price']).sum() /window['item_quantity'].sum()

# hour_mapping = {
#     22: -1, 23: -1, 
#     1: -1, 6: -1, 7: -1
# }

# df2 = df.copy()

# df2 = df2.query('item_type != "Drink" and dish_category != "Alcohol"')

# model_data = (df2
#               .assign(
#                   hour_of_day = lambda df: df.index.to_series().dt.hour.replace(hour_mapping).astype("category"),
#                   day_of_week = lambda df: df.index.to_series().dt.dayofweek.astype("category"),
#                   weekend = lambda df: pd.Series(df.index.dayofweek.isin([5, 6]).astype(int), index=df.index).astype("category"),
#                   meal_period = lambda df: pd.cut(df.index.to_series().dt.hour.astype("category"), 
#                                     bins=[0, 5, 11, 16, 22, 24], 
#                                     labels=['Late', 'Breakfast', 'Lunch', 'Dinner', 'Late'], 
#                                     right=False,
#                                     ordered=False).replace({'Late': 'Dinner'}),
#                   day_of_month = lambda df: df.index.to_series().dt.day.astype("category"),
#                   month = lambda df: df.index.to_series().dt.month.astype("category"),
#                   season = lambda df: df.index.month.map(season_from_month).astype("category"),
#                   date = lambda df: df.index.to_series().dt.date.astype("category").cat.codes)
#               .reset_index()
#               .set_index('unique_id')
#               .join([(df2
#                       .query('~vegetarian')
#                       ['item_price']            
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('~vegetarian')['unique_id'].values, axis=0)
#                       .rename('meat_window_price')),
#                      (df2
#                       .query('~vegetarian')
#                       ['item_quantity'].rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('~vegetarian')['unique_id'].values, axis=0)
#                       .rename('meat_window_quantity')),
#                      (df2
#                       .query('vegetarian')
#                       ['item_price'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('vegetarian')['unique_id'].values, axis=0)
#                       .rename('vegetarian_window_price')),
#                      (df2
#                       .query('vegetarian')
#                       ['item_quantity']
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('vegetarian')['unique_id'].values, axis=0)
#                       .rename('vegetarian_window_quantity')),
#                      (df2
#                       .query('vegan')
#                       ['item_price'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('vegan')['unique_id'].values, axis=0)
#                       .rename('vegan_window_price')),
#                      (df2
#                       .query('vegan')
#                       ['item_quantity'] 
#                       .rolling(f'{lookback_period}{lookback_unit}')
#                       .sum()
#                       .shift(1)
#                       .bfill()
#                       .reset_index(drop=True)
#                       .set_axis(df2.query('vegan')['unique_id'].values, axis=0)
#                       .rename('vegan_window_quantity'))
#                     ],
#                     how='left')
#               .reset_index()
#               .set_index('created_at')
#               .assign(
#                   vegan_window_price = lambda df: df['vegan_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegan_window_quantity = lambda df: df['vegan_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegetarian_window_price = lambda df: df['vegetarian_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegetarian_window_quantity = lambda df: df['vegetarian_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   meat_window_price = lambda df: df['meat_window_price'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   meat_window_quantity = lambda df: df['meat_window_quantity'].bfill().mask(df.index[-1] <= df.index).ffill(),
#                   vegan_window_avg = lambda df: df['vegan_window_price'] / df['vegan_window_quantity'],
#                   vegetarian_window_avg = lambda df: df['vegetarian_window_price'] / df['vegetarian_window_quantity'],
#                   meat_window_avg = lambda df: df['meat_window_price'] / df['meat_window_quantity'],
#                   vegan_outcome = lambda df: 1*df['vegan'],
#                   vegetarian_outcome = lambda df: 1*df['vegetarian']
#               ))

# model_data = model_data.loc[model_data.index.repeat(model_data['item_quantity'])]
# model_data['item_quantity'] = 1

# # majority = model_data.query('~vegan')
# # minority = model_data.query('vegan')

# # from sklearn.utils import resample

# # majority_downsampled = resample(majority, 
# #                                 replace=False, 
# #                                 n_samples=len(majority),
# #                                 random_state=1)
# # balanced_data = pd.concat([majority_downsampled, minority])

# model_data['vegan'].value_counts()

In [ ]:
# interaction_predictors = [
#     'meat_window_avg',
#     #'vegetarian_window_avg',
#     'vegan_window_avg',
# ]

# time_predictors = [
#     #'hour_of_day',
#     'meal_period',
#     'weekend',
#     'day_of_week',
#     #'day_of_month',
#     'month',
#     'season',
#     'date',
# ]

# # Split the data into training and testing sets
# train_size = model_data.shape[0] // 2

# train_data = (model_data
#               .dropna(subset=interaction_predictors + time_predictors)
#               .reset_index()
#               .iloc[:train_size, :])
# test_data = (model_data
#              .dropna(subset=interaction_predictors + time_predictors)
#              .reset_index()
#              .iloc[train_size:, :])

# # Fit logistic regression model
# formula = ['vegan_outcome ~ ',
#            # '(',' + '.join(interaction_predictors),')**2',
#            # ' + ',
#            ' + '.join(interaction_predictors), 
#            ' + vegan_window_avg:meat_window_avg + vegetarian_window_avg:meat_window_avg',
#            ' + ',
#            ' + '.join(time_predictors)
#            ]
# logit_model = smf.logit(''.join(formula), train_data)
# logit_fit = logit_model.fit(maxiter=200)

# # Fit ARIMA model to residuals

# train_data['pred'] = logit_fit.predict(train_data)
# # train_data['residuals'] = train_data['vegan_outcome'] - train_data['pred']
# # arima_model = ARIMA(train_data['residuals'], order=(1, 0, 1)).fit()
# # print(arima_model.summary())
# # print(logit_fit.summary())

# # Predict for training and testing sets
# #train_data['arima_adjusted_pred'] = arima_model.fittedvalues + train_data['pred']
# test_data['pred'] = logit_fit.predict(test_data)
# #test_data['arima_forecast'] = arima_model.get_forecast(steps=len(test_data)).predicted_mean
# #test_data['arima_adjusted_pred'] = test_data['pred'] + test_data['arima_forecast']

# # Plot function for resampling and visualization
# def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
#     resampled_pred = data.resample(freq)['pred'].mean()
#     resampled_actual = data.resample(freq)['vegan_outcome'].mean()

#     if start_date and end_date:
#         resampled_pred = resampled_pred.loc[start_date:end_date]
#         resampled_actual = resampled_actual.loc[start_date:end_date]

#     resampled_pred.plot(color='orange', label='Predicted')
#     resampled_actual.plot(color='blue', alpha=0.3, label='Actual')

#     plt.title(title)
#     plt.legend()
#     plt.show()

# # Resample and plot results for training data
# train_result = train_data.set_index('created_at')
# plot_resampled(train_result, '7D', title="Training Data: Weekly Resampled Predictions")
# #plot_resampled(train_result, '7D', start_date='2020', end_date='2021', title="Training Data: Weekly (2020-2021)")
# #plot_resampled(train_result, '30D', title="Training Data: Monthly Resampled Predictions")

# # Resample and plot results for testing data
# test_result = test_data.set_index('created_at')
# plot_resampled(test_result, '7D', title="Testing Data: Weekly Resampled Predictions")
# #plot_resampled(test_result, '7D', start_date='2020', end_date='2021', title="Testing Data: Weekly (2020-2021)")
# #plot_resampled(test_result, '30D', title="Testing Data: Monthly Resampled Predictions")

